In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
ds = pd.read_csv("https://raw.githubusercontent.com/datasets/breast-cancer/refs/heads/main/data/breast-cancer.csv")
ds.head()

,age,mefalsepause,tumor-size,inv-falsedes,falsede-caps,deg-malig,breast,breast-quad,irradiat,class
0,40-49,premefalse,15-19,0-2,True,3,right,left_up,False,recurrence-events
1,50-59,ge40,15-19,0-2,False,1,right,central,False,false-recurrence-events
2,50-59,ge40,35-39,0-2,False,2,left,left_low,False,recurrence-events
3,40-49,premefalse,35-39,0-2,True,3,right,left_low,True,false-recurrence-events
4,40-49,premefalse,30-34,3-5,True,2,left,right_up,False,recurrence-events


In [3]:
X = pd.get_dummies(ds.iloc[:, :-1],drop_first=True)
le = LabelEncoder()
y = le.fit_transform(ds.iloc[:, -1])

In [4]:
X_train, X_test, y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [5]:
#Standard scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [6]:
#CONVERTING NUMPY ARRAYS TO PYTORCH TENSORS
import torch
X_train = torch.from_numpy(X_train)
X_test = torch.from_numpy(X_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

In [11]:
X_train = X_train.float()
X_test  = X_test.float()
y_train = y_train.float()
y_test  = y_test.float()

In [12]:
#DEFINING A MODEL
import torch.nn as nn

class mysimpleNN(nn.Module) :
    
  def __init__(self,num_features) :
      super().__init__() 
      self.network = nn.Sequential(
          nn.Linear(num_features,1) ,
          nn.Sigmoid() )
    #self.weights = torch.rand(X.shape[1],1,dtype = torch.float64,requires_grad=True) #here we writing (X.shape[1],1)--> for defining shape of rand function like 30 rows and 1 column

    #self.bias = torch.zeros(1,dtype = torch.float64,requires_grad=True)

  def forward(self,features) :
      out = self.network(features)
      return out 
      #z = torch.matmul(X,self.weights) + self.bias
      #y_pred = torch.sigmoid(z)
      #return y_pred



  #def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    #epsilon = 1e-7
    #y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Binary Cross Entropy Loss
    #loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()

    #return loss


In [13]:
#LEARNING PARAMETERS
learning_rate = 0.1
epochs = 25

In [19]:
loss_function = nn.BCELoss()

In [21]:
# TRAINING PIPELINE

#CREATING MODEL
model = mysimpleNN(X_train.shape[1])

#DEFINE LOOP
for epoch in range(epochs) :

  #forward
  y_pred = model(X_train)

  #loss
  loss = loss_function(y_pred,y_train.view(-1,1))
    
#{view(-1, 1)
#It reshapes a tensor (changes its dimensions)}
# -1 -->	PyTorch automatically decides this size
# 1 -->	Make it 1 column


  #backward
  loss.backward()

  #PARAMETERS UPDATING
  with torch.no_grad() :
    model.network[0].weight -= learning_rate * model.network[0].weight.grad
    model.network[0].bias -= learning_rate * model.network[0].bias.grad

    # Reset gradients
    model.network[0].weight.grad.zero_()
    model.network[0].bias.grad.zero_()

  print(f'Epoch:{epoch+1} , Loss:{loss}')



Epoch:1 , Loss:0.7136054039001465
Epoch:2 , Loss:0.7003635764122009
Epoch:3 , Loss:0.6881451606750488
Epoch:4 , Loss:0.6768648624420166
Epoch:5 , Loss:0.6664429903030396
Epoch:6 , Loss:0.6568063497543335
Epoch:7 , Loss:0.6478869915008545
Epoch:8 , Loss:0.6396227478981018
Epoch:9 , Loss:0.6319568753242493
Epoch:10 , Loss:0.6248376369476318
Epoch:11 , Loss:0.6182178258895874
Epoch:12 , Loss:0.612054705619812
Epoch:13 , Loss:0.6063095331192017
Epoch:14 , Loss:0.6009469628334045
Epoch:15 , Loss:0.5959351062774658
Epoch:16 , Loss:0.591245174407959
Epoch:17 , Loss:0.5868508815765381
Epoch:18 , Loss:0.5827284455299377
Epoch:19 , Loss:0.5788564085960388
Epoch:20 , Loss:0.5752151608467102
Epoch:21 , Loss:0.5717869997024536
Epoch:22 , Loss:0.5685556530952454
Epoch:23 , Loss:0.5655065178871155
Epoch:24 , Loss:0.5626262426376343
Epoch:25 , Loss:0.559902548789978


In [22]:
#MODEL EVALUATION
with torch.no_grad() :
  y_pred = model(X_test)
  y_pred = (y_pred>0.7).float()
  accuracy = (y_pred == y_test).float().mean()
  print(f'Accuracy:{accuracy}')

Accuracy:0.6919008493423462
